# Train CustomLLM on a Colab GPU

This notebook installs dependencies, obtains the repository, downloads WikiText-2, retrains the 4,096-token BPE tokenizer, trains `CustomLLM` for 10 epochs with CUDA AMP, and downloads `best_model.pt`.

Select a GPU runtime in **Runtime > Change runtime type** before starting.

In [ ]:
!pip -q install torch tokenizers

import os
import sys
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime in Colab before training.')
print('GPU:', torch.cuda.get_device_name(0))

## Get the repository

Set `REPO_URL` to the GitHub repository URL. If it is left empty, the next cell opens a Colab upload dialog for a ZIP containing the repository files.

In [ ]:
from pathlib import Path
import shutil

REPO_URL = ''  # Example: 'https://github.com/your-user/test_llm.git'
WORKSPACE = Path('/content/test_llm')

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

if REPO_URL:
    !git clone {REPO_URL} {WORKSPACE}
else:
    from google.colab import files
    print('Upload a ZIP of the repository, then rerun this cell if needed.')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
    if zip_name is None:
        raise ValueError('Upload a repository ZIP or set REPO_URL.')
    import zipfile
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall('/content')
    extracted = Path('/content') / Path(zip_name).stem
    if extracted.exists() and extracted != WORKSPACE:
        extracted.rename(WORKSPACE)

os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
print('Workspace:', Path.cwd())

In [ ]:
# Download and clean WikiText-2 using the repository script.
!python download_data.py

# Force a clean tokenizer retrain at the requested 4,096 vocabulary size.
tokenizer_path = Path('model/tokenizer.json')
if tokenizer_path.exists():
    tokenizer_path.unlink()
print('Tokenizer reset:', not tokenizer_path.exists())

In [ ]:
from model.bpe_data import train_or_load_tokenizer

tokenizer = train_or_load_tokenizer(
    'model/input.txt',
    'model/tokenizer.json',
    vocab_size=4096,
)
print('BPE vocabulary size:', tokenizer.get_vocab_size())

## GPU training

This cell uses `torch.cuda.amp.autocast()` and `GradScaler` and saves the best validation checkpoint to `model/weights/best_model.pt`.

In [ ]:
import math
from torch import nn
from torch.optim import AdamW
from model.architecture import CustomLLM
from model.bpe_data import CausalLanguageModelDataset, create_dataloader

DEVICE = torch.device('cuda')
EPOCHS = 10
BATCH_SIZE = 16
SEQUENCE_LENGTH = 256
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01

dataset = CausalLanguageModelDataset.from_file(
    'model/input.txt', tokenizer, sequence_length=SEQUENCE_LENGTH
)
validation_size = max(1, int(len(dataset) * 0.2))
training_size = len(dataset) - validation_size
training_dataset, validation_dataset = torch.utils.data.random_split(
    dataset, [training_size, validation_size],
    generator=torch.Generator().manual_seed(42),
)
training_loader = create_dataloader(training_dataset, BATCH_SIZE, shuffle=True)
validation_loader = create_dataloader(validation_dataset, BATCH_SIZE, shuffle=False)

model = CustomLLM(
    tokenizer=tokenizer,
    block_size=256,
    d_model=128,
    n_layers=2,
).to(DEVICE)

decay_params = [p for p in model.parameters() if p.requires_grad and p.ndim == 2]
no_decay_params = [p for p in model.parameters() if p.requires_grad and p.ndim != 2]
optimizer = AdamW([
    {'params': decay_params, 'weight_decay': WEIGHT_DECAY},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()
total_steps = EPOCHS * max(1, len(training_loader))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
best_validation_loss = float('inf')
checkpoint_path = Path('model/weights/best_model.pt')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    train_loss_total = 0.0
    for batch_index, batch in enumerate(training_loader, start=1):
        input_ids = batch['input_ids'].to(DEVICE)
        targets = batch['labels'].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits = model(input_ids)
            loss = loss_fn(logits.view(-1, tokenizer.get_vocab_size()), targets.view(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss_total += loss.item()
        if batch_index % 100 == 0:
            print(f'epoch {epoch + 1}/{EPOCHS}, batch {batch_index}/{len(training_loader)}, loss={loss.item():.4f}', flush=True)

    model.eval()
    validation_loss_total = 0.0
    with torch.no_grad():
        for batch in validation_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            targets = batch['labels'].to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(input_ids)
                validation_loss_total += loss_fn(
                    logits.view(-1, tokenizer.get_vocab_size()), targets.view(-1)
                ).item()
    train_loss = train_loss_total / max(1, len(training_loader))
    validation_loss = validation_loss_total / max(1, len(validation_loader))
    print(f'epoch {epoch + 1}/{EPOCHS}, train_loss={train_loss:.4f}, validation_loss={validation_loss:.4f}', flush=True)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'validation_loss': validation_loss,
            'epoch': epoch + 1,
        }, checkpoint_path)
        print('saved best checkpoint to', checkpoint_path, flush=True)

print('Best validation loss:', best_validation_loss)
print('Checkpoint exists:', checkpoint_path.exists())

In [ ]:
# Download the trained checkpoint to your local machine.
from google.colab import files

if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint was not created: {checkpoint_path}')
files.download(str(checkpoint_path))